In [1]:
import os, sys, importlib
sys.path.append(os.path.abspath(".."))
import json
import pandas as pd
import time
from tqdm import tqdm
import pdfplumber
from collections import defaultdict

# my utils
from utils import extract_from_pdf
from utils import parse_notice

In [2]:
path_pdf="../data\catalogue_of_romance_vol1.pdf"

In [ ]:
import pdfplumber
import re
from collections import defaultdict
import statistics

# -------------------------
# 1️⃣ extract words
# -------------------------
def extract_words_per_page(path_pdf, page_num):

    with pdfplumber.open(path_pdf) as pdf:

        page = pdf.pages[page_num-1]

        words = page.extract_words(extra_attrs=["size"])

    return words

def extract_words_start_end(path_pdf, start_page, end_page):

    with pdfplumber.open(path_pdf) as pdf:

        page = pdf.pages[start_page-1:end_page]

        words = page.extract_words(extra_attrs=["size"])

    return words


# -------------------------
# 2️⃣ group words → lines
# -------------------------
def group_words_to_lines(words):

    lines = defaultdict(list)

    for w in words:

        if not isinstance(w, dict):
            continue

        if "top" not in w:
            continue

        y = round(w["top"], 1)
        lines[y].append(w)

    sorted_lines = sorted(lines.items(), key=lambda x: x[0])

    result = []

    for y, ws in sorted_lines:

        ws = sorted(ws, key=lambda x: x["x0"])

        text = " ".join(w["text"] for w in ws)

        avg_size = sum(w.get("size", 0) for w in ws) / len(ws)

        x0 = ws[0]["x0"]
        x1 = ws[-1]["x1"]
        center = (x0 + x1) / 2

        result.append({
            "text": text.strip(),
            "size": avg_size,
            "center": center,
            "words": ws
        })

    return result


# -------------------------
# 3️⃣ detect notices
# -------------------------


def has_letter_and_digit(s):
    return any(c.isalpha() for c in s) and any(c.isdigit() for c in s)



def extract_notices(lines, follow_lines=3):
    prefix=['royal',
            'cotton',
            'harley',
            'lansdowne',
            'arundel',
            'burney',
            'sloane',
            'add',# 'additional'
            'egerton'
        ]

    notices = {}

    # 自动估计正文字体大小
    # sizes = [l["size"] for l in lines if l["size"] > 0]
    # body_size = statistics.median(sizes) if sizes else 10
    # print(f"body font size:{body_size}\n")
    
    # page_center = page_width / 2

    for i, line in enumerate(lines):

        text = line["text"]

        # -------------------------
        # 标题判断（多条件组合）
        # -------------------------
        # is_centered = abs(line["center"] - page_center) < page_width * 0.15
        is_large = line["size"] > 10 #body_size #* 1.2
        # is_short = len(text.split()) <= 10
        startswith_coll=text.lower().startswith(tuple(prefix))
        is_mixed=has_letter_and_digit(text)
        # print(f'is large:{is_large}; startswith_coll:{startswith_coll}; is_mixed: {is_mixed}!\n')
        # if is_centered and is_large and is_short:
        
        
        if is_large and startswith_coll and is_mixed:
            mss=text.lower().strip()
            # print(f"valid line: {line}\n")  
            block = []
            # -------------------------
            # 抓取后续行
            # -------------------------
            for j in range(1, follow_lines + 1):

                if i + j < len(lines):
                    next_text = lines[i + j]["text"]

                    # 跳过空行
                    if not next_text.strip():
                        continue
                    block.append(next_text)
            # block_s=" ".join(block)

            notices[mss]=" ".join(block) #.append(block)

    return notices



import re
def eliminate_ponc(s):
    clean_s=re.sub(r'[^\w\s]','',s)
    clean_s=clean_s.strip().lower()
    return clean_s


def roman_to_int(s):
    
    ROMAN_MAP = {
        'i':1,'v':5,'x':10,'l':50,'c':100
    }
    s = s.lower()
    total = 0
    prev = 0

    for ch in reversed(s):
        val = ROMAN_MAP.get(ch, 0)
        if val < prev:
            total -= val
        else:
            total += val
        prev = val

    return total


def fix_roman_ordinal(text):
    
    def repl(m):
        roman = m.group(1)
        suffix = m.group(2)

        num = roman_to_int(roman)

        return f"{num}{suffix}"

    return re.sub(r'\b([ivxlc]+)(th|st|nd|rd)\b', repl, text, flags=re.I)


def get_notice_info(notices):
    notice_info=[]
    for mss, notice in notices.items():
        if notice.count(";")>=2:
            # print(mss)
            # print(f"{notice.count(';')} in '{notice}' \n")
            
            try:
                #---init---
                material,cent, size=None, None, None
                
                notice_el=notice.split(";")
                # print(notice_el)
                
                # #---materia---
                material=notice_el[0].lower().strip()
                # print(f"material: {material}")

                # #---cent & size---   
                if 'cent' in notice_el[1]:#如果有cent，按cent切分
                    cent=eliminate_ponc(notice_el[1].split("cent")[0])      
                    size=eliminate_ponc(notice_el[1].split("cent")[1])

                else: # 没有则找大写切分呢
                    first_upper_idx = next((i for i, c in enumerate(notice_el[1]) if c.isupper()), None)
                    if first_upper_idx:# 若有大写
                        # print(f"first_upper_idx:{first_upper_idx}; first supper :{notice_el[1][first_upper_idx]}")
                        cent=eliminate_ponc(notice_el[1][:first_upper_idx])
                        size=eliminate_ponc(notice_el[1][first_upper_idx:])
        
                info={
                    "mss":mss,
                    "mateiral":material,
                    "cent":cent,
                    "size":size,
                    }
            
                notice_info.append(info)    
            except Exception as e:
                print(f"[error] {e} \n")
                continue
    return notice_info





# -------------------------
# 4️⃣ full pipeline
# -------------------------
def extract_notice_pipeline(path_pdf, start_page, end_page):
    result=[]
    for p in range(start_page, end_page+1):
        print(f"page {p}".center(100, '-'))
        words_per_page= extract_words_per_page(path_pdf, page_num=p)
        lines = group_words_to_lines(words_per_page)
        # print(lines)
        notices = extract_notices(lines, follow_lines=2)
        print(notices)
        
        notice_info=get_notice_info(notices)
        print(notice_info)    
        result.extend(notice_info)
    return result


In [91]:
result=extract_notice_pipeline(path_pdf, 378, 379)
result

----------------------------------------------page 378----------------------------------------------
{'royal 20. d. iii.': 'Vellum ; b eginning of the xivth cent. Folio ; f f. 2 07, in double columns, having 42 lines to a column. With coloured initials.', 'additional 10,293, f. 129, col. 1, boyal 19. c. xin, f. 81 b, and': 'Lansdowne 757, which ends with the same passage. Harley 4410.', 'lansdowne 757, which ends with the same passage.': 'Harley 4410. Vellum; xrvth cent Folio; ff. 168, in double columns, having 45 lines', 'harley 4410.': 'Vellum; xrvth cent Folio; ff. 168, in double columns, having 45 lines to a colimm. With coloured initials, and with an illuminated initial and'}
[{'mss': 'royal 20. d. iii.', 'mateiral': 'vellum', 'cent': 'b eginning of the xivth', 'size': 'folio'}, {'mss': 'lansdowne 757, which ends with the same passage.', 'mateiral': 'harley 4410. vellum', 'cent': 'xrvth', 'size': 'folio'}, {'mss': 'harley 4410.', 'mateiral': 'vellum', 'cent': 'xrvth', 'size': 'fol

[{'mss': 'royal 20. d. iii.',
  'mateiral': 'vellum',
  'cent': 'b eginning of the xivth',
  'size': 'folio'},
 {'mss': 'lansdowne 757, which ends with the same passage.',
  'mateiral': 'harley 4410. vellum',
  'cent': 'xrvth',
  'size': 'folio'},
 {'mss': 'harley 4410.',
  'mateiral': 'vellum',
  'cent': 'xrvth',
  'size': 'folio'},
 {'mss': 'royal 14: e. iii. flf. 89-i6i b.',
  'mateiral': 'vellum',
  'cent': 'early xivth',
  'size': 'large folio'}]

In [92]:
start_page=26
end_page=982


start_time=time.time()

result=extract_notice_pipeline(path_pdf, start_page, end_page)
end_time=time.time()
print(f"[done] {len(result)} notices from p{start_page} to p{end_page} in {end_time-start_time:.2f} sec!")

----------------------------------------------page 26-----------------------------------------------
{'royal 16. c. zxiii. ff. 2-69 b.': 'Vellxun; xvth cent. Small Qnarto; ff. 68, with 21 lines to a page. Headings, glosses, and initials, in red. Formerly belonged to Sir Bobert'}
[{'mss': 'royal 16. c. zxiii. ff. 2-69 b.', 'mateiral': 'vellxun', 'cent': 'xvth', 'size': 'small qnarto'}]
----------------------------------------------page 27-----------------------------------------------
{'royal 16* c. iv. a. b.': 'Paper ; t wo vols., a.d. 1560 and 1565. Small Quarto ; f f. 4 6, and ff. 98 ; t he full pages of Vol. i. h aving 35 to 36 and those of Vol. n. 24 to 27 lines of verse.'}
[{'mss': 'royal 16* c. iv. a. b.', 'mateiral': 'paper', 'cent': 't wo vols ad 1560 and 1565', 'size': 'small quarto'}]
----------------------------------------------page 28-----------------------------------------------
{}
[]
----------------------------------------------page 29----------------------------------

In [131]:
result

[{'mss': 'royal 16. c. zxiii. ff. 2-69 b.',
  'mateiral': 'vellxun',
  'cent': 'xvth',
  'size': 'small qnarto',
  'mss_index': 'royal 16. c. xxiii. ff. 2-69 b'},
 {'mss': 'royal 16* c. iv. a. b.',
  'mateiral': 'paper',
  'cent': 't wo vols ad 1560 and 1565',
  'size': 'small quarto',
  'mss_index': 'royal 16. c. iv. a. b.'},
 {'mss': 'royal 16. d. iii. a. b.',
  'mateiral': 'paper',
  'cent': 'zvith',
  'size': '',
  'mss_index': 'royal 16. d. iii. a. b.'},
 {'mss': 'harley 6662. «. i-56.',
  'mateiral': 'paper',
  'cent': 'xvth',
  'size': 'quarto',
  'mss_index': 'harley 5662. ff. 1-56'},
 {'mss': 'additional 15,429.',
  'mateiral': 'paper',
  'cent': 'xvth',
  'size': 'octavo',
  'mss_index': 'additional add. 15,429'},
 {'mss': 'harley 3514.',
  'mateiral': 'vellum',
  'cent': 'xvth',
  'size': 'octavo',
  'mss_index': 'harley 3514'},
 {'mss': 'cotton^ vespasian b. zzv. ff. 98b-ii7.',
  'mateiral': 'vellum',
  'cent': 'xnth',
  'size': 'quarto',
  'mss_index': 'cotton vespasian b.

In [130]:
## save
path_result="../results/vol1/vol1-notices.json"
with open(path_result, 'w', encoding='utf-8')as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
    

## CLEAN & UNIFIER

In [ ]:
## unifier les indice mss! basé sur index_table_clean (corrigée manuellement!)
 
path_index_table=r"../results\vol1\index_table_clean.json"
with open (path_index_table, 'r', encoding='utf-8') as f :
    index_table=json.load(f)
index_table

{'Royal': [{'name': 'Historia Regum Britanniae',
   'page': 227,
   'mss': '4. C. xi. ff. 222-249'},
  {'name': "Turpin's Chronicle",
   'page': 583,
   'mss': '4. C. xi. ff. 280-286 b'},
  {'name': 'Prophecy of Merlin Silvester',
   'page': 313,
   'mss': '5. F. xv. ff. 3 b'},
  {'name': 'Dares Phrygius', 'page': 17, 'mss': '6. C. viii. ff. 123-133 b'},
  {'name': 'Letters between Alexander and Dindimus',
   'page': 138,
   'mss': '6. E. iii. ff. 111 b-112 b'},
  {'name': 'Letters between Alexander and Dindimus',
   'page': 137,
   'mss': '7. A. i. ff. 68 b-69 b'},
  {'name': 'Alexandreis', 'page': 98, 'mss': '8. B. iv. ff. 19-71 b'},
  {'name': 'Prophecies of Merlin',
   'page': 297,
   'mss': '8. D. iii. ff. 160 b-163'},
  {'name': 'Guy of Warwick', 'page': 485, 'mss': '8. F. ix. ff. 105-159'},
  {'name': 'Prophecy on Scotland', 'page': 327, 'mss': '9. B. ix. ff. 2'},
  {'name': 'Dares Phrygius', 'page': 20, 'mss': '10. A. x. ff. 188-192 b'},
  {'name': 'Alexander the Great',
   'pa

In [ ]:
list_coll_mss=[]
for coll, list_mss_dict in index_table.items():
    list_mss=[_["mss"].lower().strip() for _ in list_mss_dict]
    
    for mss in list_mss:
        if not mss.startswith(coll.lower()):
            mss=f"{coll.lower()} {mss}"
        list_coll_mss.append(mss) 
    # break # chq coll
print(len(list_coll_mss))
print(list_coll_mss[:10])


508
['royal 4. c. xi. ff. 222-249', 'royal 4. c. xi. ff. 280-286 b', 'royal 5. f. xv. ff. 3 b', 'royal 6. c. viii. ff. 123-133 b', 'royal 6. e. iii. ff. 111 b-112 b', 'royal 7. a. i. ff. 68 b-69 b', 'royal 8. b. iv. ff. 19-71 b', 'royal 8. d. iii. ff. 160 b-163', 'royal 8. f. ix. ff. 105-159', 'royal 9. b. ix. ff. 2']


In [123]:
from rapidfuzz import process, fuzz
def get_el_by_fuzzy_research(request, contexts, cutoff=70):
    # for r in list_research:
    # 若contexts中铀元素包含待查询内容
    el_contains=[c for c in contexts if request in c]
    if el_contains :
        el=el_contains[0].strip()
        return request
        
    #若没有匹配，进行模糊搜索：
    result=process.extractOne(
        request, 
        contexts,
        scorer=fuzz.ratio,
        score_cutoff=cutoff
    )
    if result:            
        # print(result[0])
        idx_result=contexts.index(result[0])
        el=contexts[idx_result].strip()
        # print(f"fuzzy result:{result[0]}")   
        return result[0]
    
    else:
        # print(f"no match for '{r}'!")
        return None


In [ ]:
##  在notices result中增加一个字段 mss_match (in index_table) 作为index
result_clean=[]
for i, notice in enumerate(result):
    mss_ocr=notice["mss"]
    mss_index=get_el_by_fuzzy_research(mss_ocr, list_coll_mss, cutoff=70)
    # print("mss_index: ",mss_index)
    # print('context: ',context)
    notice.update({"mss_index":mss_index})
    result_clean.append(notice)
    

In [ ]:
result_clean

[{'mss': 'royal 16. c. zxiii. ff. 2-69 b.',
  'mateiral': 'vellxun',
  'cent': 'xvth',
  'size': 'small qnarto',
  'mss_index': 'royal 16. c. xxiii. ff. 2-69 b'},
 {'mss': 'royal 16* c. iv. a. b.',
  'mateiral': 'paper',
  'cent': 't wo vols ad 1560 and 1565',
  'size': 'small quarto',
  'mss_index': 'royal 16. c. iv. a. b.'},
 {'mss': 'royal 16. d. iii. a. b.',
  'mateiral': 'paper',
  'cent': 'zvith',
  'size': '',
  'mss_index': 'royal 16. d. iii. a. b.'},
 {'mss': 'harley 6662. «. i-56.',
  'mateiral': 'paper',
  'cent': 'xvth',
  'size': 'quarto',
  'mss_index': 'harley 5662. ff. 1-56'},
 {'mss': 'additional 15,429.',
  'mateiral': 'paper',
  'cent': 'xvth',
  'size': 'octavo',
  'mss_index': 'additional add. 15,429'},
 {'mss': 'harley 3514.',
  'mateiral': 'vellum',
  'cent': 'xvth',
  'size': 'octavo',
  'mss_index': 'harley 3514'},
 {'mss': 'cotton^ vespasian b. zzv. ff. 98b-ii7.',
  'mateiral': 'vellum',
  'cent': 'xnth',
  'size': 'quarto',
  'mss_index': 'cotton vespasian b.

In [132]:
path_result_clean=r"../results\vol1\vol1-notices.json"
with open(path_result_clean, "w", encoding="utf-8") as f:
    json.dump(result_clean, f, indent=2, ensure_ascii=False)